In [14]:
# ============================================
# Moroccan Legal RAG - GPU Embedding Service
# Environment Setup
# ============================================

# Install dependencies with compatible versions
!pip install -q fastapi uvicorn==0.30.6 sentence-transformers nest-asyncio

# Download Cloudflare tunnel client
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

# Make it executable
!chmod +x cloudflared

# Required for running FastAPI inside Colab
import nest_asyncio
nest_asyncio.apply()

print("✓ Environment setup complete.")
print(f"  - fastapi, uvicorn==0.30.6, sentence-transformers installed")
print(f"  - cloudflared downloaded")

cloudflared: Text file busy
✓ Environment setup complete.
  - fastapi, uvicorn==0.30.6, sentence-transformers installed
  - cloudflared downloaded


In [ ]:
# ============================================
# Load BAAI/bge-m3 on GPU
# ============================================

from sentence_transformers import SentenceTransformer
import torch
import time

MODEL_NAME = "BAAI/bge-m3"

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU not available.\n"
        "Go to: Runtime → Change runtime type → Select T4 GPU → Save\n"
        "Then re-run this cell."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print()

# Load model
print(f"Loading model: {MODEL_NAME}...")
start = time.time()

model = SentenceTransformer(MODEL_NAME)
model = model.to("cuda")

elapsed = time.time() - start
print(f"✓ Model loaded in {elapsed:.1f}s")
print(f"  Device: {model.device}")
print(f"  Dimension: {model.get_sentence_embedding_dimension()}")


CUDA available: True
GPU: Tesla T4

Loading model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded in 12.4s
  Device: cuda:0
  Dimension: 1024


/tmp/ipykernel_8588/3287475750.py:33: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Dimension: {model.get_sentence_embedding_dimension()}")


In [16]:
# Warm-up
print("\nWarming up...")
warmup_start = time.time()
_ = model.encode(
    [
        "اختبار",              # Arabic
        "test juridique",      # French
        "legal test"           # English
    ],
    normalize_embeddings=True
)
warmup_elapsed = time.time() - warmup_start
print(f"✓ Warm-up complete in {warmup_elapsed:.2f}s")
print("\n✅ Model ready for requests.")


Warming up...
✓ Warm-up complete in 0.14s

✅ Model ready for requests.


In [ ]:
# ============================================
# FastAPI Application
# ============================================

from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

app = FastAPI(
    title="Moroccan Legal RAG - Embedding Service",
    version="1.0.0"

)

# Request/Response models
class EmbedRequest(BaseModel):
    texts: list[str]
    batch_size: int = 32

class EmbedResponse(BaseModel):
    embeddings: list[list[float]]
    dimension: int
    count: int
    time_ms: float

# Routes
@app.get("/")
async def root():
    return {
        "message": "Moroccan Legal RAG Embedding Service",
        "docs": "/docs"
    }

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_NAME,
        "device": str(model.device),
        "dimension": model.get_sentence_embedding_dimension(),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        )
    }

@app.post("/embed", response_model=EmbedResponse)
async def embed(request: EmbedRequest):
    start = time.time()

    embeddings = model.encode(
        request.texts,
        batch_size=request.batch_size,
        normalize_embeddings=True,
        show_progress_bar=len(request.texts) > 50,
        convert_to_numpy=True
    )

    elapsed = (time.time() - start) * 1000

    return EmbedResponse(
        embeddings=embeddings.tolist(),
        dimension=int(embeddings.shape[1]),
        count=len(request.texts),
        time_ms=round(elapsed, 2)
    )

print("✓ FastAPI app created.")
print("  Routes: /  /health  /embed  /docs")

✓ FastAPI app created.
  Routes: /  /health  /embed  /docs


In [ ]:
# ============================================
# Start Cloudflare Tunnel + FastAPI Server
# ============================================

import subprocess
import threading
import uvicorn
import re
import time
import os

def start_tunnel():
    """Start cloudflared quick tunnel in background."""
    time.sleep(5)  # Wait for uvicorn to start

    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line.strip())

        # Extract tunnel URL
        match = re.search(
            r"https://[-a-zA-Z0-9]+\.trycloudflare\.com",
            line
        )

        if match:
            print()
            print("=" * 60)
            print("✅ PUBLIC TUNNEL URL:")
            print(f"   {match.group(0)}")
            print("=" * 60)
            print()
            print("Copy this into your local .env file:")
            print(f'EMBEDDING_SERVICE_URL={match.group(0)}')
            print()
            print("Test it:")
            print(f"  curl {match.group(0)}/health")
            print()

# Start tunnel in background thread
tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

# Configure uvicorn to run in notebook
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

print("Starting FastAPI server...")
print("Tunnel URL will appear below once connected.")
print("-" * 60)

# Run server (this blocks the cell - expected behavior)
await server.serve()

Starting FastAPI server...
Tunnel URL will appear below once connected.
------------------------------------------------------------


INFO:     Started server process [8588]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-05-16T16:09:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-16T16:09:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-16T16:09:06Z INF +--------------------------------------------------------------------------------------------+
2026-05-16T16:09:06Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-16T16:09:06Z INF |  https://pop-incurred-bedding-september.trycloudflare.

/tmp/ipykernel_8588/3071799857.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "dimension": model.get_sentence_embedding_dimension(),


INFO:     102.97.203.31:0 - "POST //embed HTTP/1.1" 404 Not Found
INFO:     102.97.203.31:0 - "POST /embed HTTP/1.1" 200 OK
